In [1]:
import glob
import shutil
files = glob.glob("../data/B*")
import re
f_name = []
for f in files:
    p = 'data/(B\d{6}).TXT'
    f_name.append(re.search(p,f).group(1))

In [2]:
len(f_name)

2

In [3]:
def mkcsv(FILE_NAME):
    with open('../data/' + FILE_NAME + '.TXT', encoding='shift-jis') as f:
        data = f.readlines()
    import re

    racers = [row.replace('\u3000','').replace('\n','') for row in data if re.match('^[1-6]\s\d+[^0-9]+\d{2}[^0-9]+\d{2}[AB]\d\s', row)]
    pattern = '^([1-6])\s+(\d+)([^0-9]+)(\d{2})([^0-9]+)(\d{2})([AB]\d{1})\s*(\d+\.\d{2})\s*(\d+\.\d{2})\s*(\d+\.\d{2})\s*(\d+\.\d+)\s*\d+\s*(\d+\.\d{2})\s*\d+\s*(\d+\.\d{2})'
    pattern_re = re.compile(pattern)

    values = [re.match(pattern_re, racer).groups() for racer in racers]
    import pandas as pd

    column = ['艇番', '選手登番', '選手名', '年齢', '支部', '体重', '級別', '全国勝率', '全国2連率', '当地勝率', '当地2連率', 'モーター2連率', 'ボート2連率']
    df = pd.DataFrame(values, columns=column)
    races = [row.replace('\u3000','').replace('\n','').replace(' ','') for row in data if re.match('^\s{3}[^0-9\s]', row)]
    import unicodedata
    races_uni = [unicodedata.normalize('NFKC', i) for i in races]
    place_code = {'桐生':'01','戸田':'02','江戸川':'03','平和島':'04','多摩川':'05','浜名湖':'06','蒲郡':'07','常滑':'08','津':'09','三国':'10','びわこ':'11','琵琶湖':'11','住之江':'12','尼崎':'13','鳴門':'14','丸亀':'15','児島':'16','宮島':'17','徳山':'18','下関':'19','若松':'20','芦屋':'21','福岡':'22','唐津':'23','大村':'24'}
    class_mapping = {'B2':1,'B1':2,'A2':3,'A1':4}


    pattern = re.compile('^.+\d+日(\d{4})年(\d+)月(\d+)日ボートレース(.+)')
    racers_repp = list([re.match(pattern, i).groups() for i in races_uni])

    racers_reppp = []
    pl =[]
    for i in racers_repp:
        for j, k in place_code.items():
            if i[3] == j:
                racers_reppp.append(str(i[0]) + '_' + str(i[1]) + '_' + str(i[2]) + '_' + str(k))
                pl.append(k)
            else:
                pass

    racers_rep_R =[]
    R = []
    for i in racers_reppp:
        for r in range(12):
            racers_rep_R.append(str(i) + '_' + str(r + 1) + 'R')
            R.append(str(r + 1) + 'R')

    df['RaceID'] = 0
    df['場所'] = 0
    df['R'] = 0

    for i in range(df.shape[0]):
        df['RaceID'][i] = racers_rep_R[i//6]
        df['R'][i] = R[i//6]
        df['場所'][i] = pl[i//72]
    df.to_csv('../csv/' + FILE_NAME +  '.csv')
    shutil.move('../data/' + FILE_NAME + '.TXT', '../done/'+ FILE_NAME + '.TXT')

In [4]:
for FILE_NAME in f_name:
    mkcsv(FILE_NAME)

/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_60437/2594155024.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RaceID'][i] = racers_rep_R[i//6]
/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_60437/2594155024.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['R'][i] = R[i//6]
/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_60437/2594155024.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-c